In [ ]:
import pandas as pd
import datasets
from datasets import Dataset
import requests

In [ ]:
def calc_op(score, rating, level_value, addition_info):
    extra_op = 0
    if addition_info == 'fullcombo':
        extra_op = 0.5
    if addition_info == 'alljustice':
        extra_op = 1
    if addition_info == 'alljusticecritical':
        extra_op = 1.25
    if score <= 1007500:
        return rating * 5 + extra_op
    else:
        return (level_value + 2) * 5 + (score - 1007500) * 0.0015 + extra_op

In [ ]:
def compute_c_from_r_s(s, r):
    """
    Given r and s, compute c based on the segmented linear function.
    """
    if s < 500000:
        # r must be 0 here; if r > 0, no solution in this segment.
        if abs(r) < 1e-9:
            return 0  # any c works, return default
        else:
            # no valid c in this segment
            pass

    if 500000 <= s < 800000:
        denom = s - 500000
        if denom > 0:
            c = 5 + 600000 * r / denom
            if c >= 5:  # since c-5 positive in formula
                return c

    if 800000 <= s < 900000:
        coeff = 0.5 + (s - 800000) / 200000
        if coeff > 0:
            c = 5 + r / coeff
            if c >= 5:
                return c

    if 900000 <= s < 925000:
        c = 5 + r - 0.00008 * (s - 900000)
        return c

    if 925000 <= s < 975000:
        c = 3 + r - 0.00006 * (s - 925000)
        return c

    if 975000 <= s < 990000:
        c = r - 0.00004 * (s - 975000)
        return c

    if 990000 <= s < 1000000:
        c = r - 0.6 - 0.00004 * (s - 990000)
        return c

    if 1000000 <= s < 1005000:
        c = r - 1.0 - 0.0001 * (s - 1000000)
        return c

    if 1005000 <= s < 1007500:
        c = r - 1.5 - 0.0002 * (s - 1005000)
        return c

    if 1007500 <= s < 1009000:
        c = r - 2.0 - 0.0001 * (s - 1007500)
        return c

    if s >= 1009000:
        c = r - 2.15
        return c

    # If no valid segment fits, raise error or return None
    raise ValueError(f"No valid segment found for s={s}, r={r}")



In [ ]:
# 公共 API：获取曲目列表
url = "https://maimai.lxns.net/api/v0/chunithm/song/list"
params = {"notes": "true"} # 可选参数

response = requests.get(url, params=params)
if response.status_code == 200:
    songs = response.json()
else:
    print(f"请求失败，状态码: {response.status_code}")

url = "https://maimai.lxns.net/api/v0/user/chunithm/player/scores"
headers = {
    "X-User-Token": "XXXXXXXXXX" # change to your personal API
}
response2 = requests.get(url, headers=headers)

# 检查响应状态
if response2.status_code == 200:
    player_data = response2.json()
else:
    print(f"请求失败，状态码: {response.status_code}")
    print(f"响应内容: {response.text}")

In [ ]:
player_data_for_op = {'song_name': [], 'level': [], 'level_value': [], 'level_index': [], 'op': [], 'addition': []}
unique_songs = list(set([record['song_name'] for record in player_data['data']]))
for song in unique_songs:
    result = [record for record in player_data['data'] if record['song_name'] == song]
    # print(result)
    max_op = 0.0
    max_level_index = -1
    max_level = '0'
    max_level_value = 0.0
    max_level_addition = None
    for i, play_data in enumerate(result):
        score = play_data['score']
        full_combo = play_data['full_combo']
        rating = play_data['rating']
        level_value = round(compute_c_from_r_s(score, rating), 1)
        if rating == 0:
            continue
        try:
            op = calc_op(score, rating, level_value, full_combo)
            if op > max_op:
                max_op = op
                max_level_index = play_data['level_index']
                max_level = play_data['level']
                max_level_value = level_value
                max_level_addition = full_combo
        except Exception as e:
            print(song)
            print(result)
            print(result_song)
        # level_value = ???
        
    player_data_for_op['song_name'].append(song)
    player_data_for_op['level'].append(max_level)
    player_data_for_op['level_value'].append(max_level_value)
    player_data_for_op['level_index'].append(max_level_index)
    player_data_for_op['op'].append(max_op)
    player_data_for_op['addition'].append(max_level_addition)
        

player_data_for_op = Dataset.from_dict(player_data_for_op)
player_data_for_op

In [ ]:
song_list_for_op = {'song_name': [], 'level': [], 'level_value': [], 'level_index': []}
for i in range(len(songs['songs'])):
    current_song = songs['songs'][i]
    if current_song['difficulties'][-1]['level_value'] == 0:
        continue
    song_list_for_op['song_name'].append(current_song['title'])
    song_list_for_op['level'].append(current_song['difficulties'][-1]['level'])
    song_list_for_op['level_value'].append(current_song['difficulties'][-1]['level_value'])
    song_list_for_op['level_index'].append(current_song['difficulties'][-1]['difficulty'])

song_list_for_op = Dataset.from_dict(song_list_for_op)
song_list_for_op

In [ ]:
player_over_power = {'level': [], 'songs': [], 'played': [], 'true_played': [], 'easier': [], 'sum_op': [], 'now_op': [], 'percentage': [], 'fc': [], 'aj': [], 'ajc': []}
level = ['10', '10+', '11', '11+', '12', '12+', '13', '13+', '14', '14+', '15', '15+']
for i in range(len(level)):
    player_over_power['level'].append(level[i])
    player_over_power['songs'].append(0)
    player_over_power['played'].append(0)
    player_over_power['sum_op'].append(0.0)
    player_over_power['now_op'].append(0.0)
    player_over_power['fc'].append(0)
    player_over_power['aj'].append(0)
    player_over_power['ajc'].append(0)
    player_over_power['easier'].append(0)
    player_over_power['true_played'].append(0)
    player_over_power['percentage'].append(0.0)

for song in song_list_for_op:
    try:
        index = level.index(song['level'])
        player_over_power['songs'][index] += 1
        player_over_power['sum_op'][index] += calc_op(1010000, 0, song['level_value'], 'alljusticecritical')
    except Exception as e:
        pass

# Why???
player_data_waht = [record for record in player_data_for_op if record['song_name'] == 'Help me, ERINNNNNN!!']
if len(player_data_waht) > 0:
    player_over_power['played'][1] += 1
    player_over_power['true_played'][1] += 1
    player_over_power['now_op'][1] += player_data_waht[0]['op']
    if player_data_waht[0]['addition'] == 'fullcombo': 
        player_over_power['fc'][1] += 1
    if player_data_waht[0]['addition'] == 'alljustice': 
        player_over_power['fc'][1] += 1
        player_over_power['aj'][1] += 1
    if player_data_waht[0]['addition'] == 'alljusticecritical': 
        player_over_power['fc'][1] += 1
        player_over_power['aj'][1] += 1
        player_over_power['ajc'][1] += 1

In [ ]:
for data in player_data_for_op:
    song_data = [record for record in song_list_for_op if record['song_name'] == data['song_name']]
    try:
        # if len(song_data) == 0:
        #     continue
        index = level.index(song_data[0]['level'])
        player_over_power['played'][index] += 1
        player_over_power['now_op'][index] += data['op']
        if song_data[0]['level_index'] == data['level_index']:
            player_over_power['true_played'][index] += 1
            if data['addition'] == 'fullcombo': 
                player_over_power['fc'][index] += 1
            if data['addition'] == 'alljustice': 
                player_over_power['fc'][index] += 1
                player_over_power['aj'][index] += 1
            if data['addition'] == 'alljusticecritical': 
                player_over_power['fc'][index] += 1
                player_over_power['aj'][index] += 1
                player_over_power['ajc'][index] += 1
        else:
             player_over_power['easier'][index] += 1
    except Exception as e:
        pass
        # print(e)
        # print(data['song_name'])
        # print(song_data)

In [ ]:
sum_all_op = 0.0
sum_played_op = 0.0

for i in range(len(level)):
    player_over_power['percentage'][i] = round(player_over_power['now_op'][i] / player_over_power['sum_op'][i] * 100, 3)
    sum_all_op += player_over_power['sum_op'][i]
    sum_played_op += player_over_power['now_op'][i]

print('Overall over power:', sum_played_op)
print('Overall over power percentage: ' + str(round(sum_played_op / sum_all_op * 100, 3)) + '%')
player_over_power_df = pd.DataFrame(player_over_power)
player_over_power_df